# Model Derivation: AI Infrastructure Investment under Regime Uncertainty

This notebook provides a complete symbolic derivation of the core model in
*Investing in Artificial General Intelligence*, using SymPy for computer algebra
verification. The derivation follows the structure of the paper's model section
and the `symbolic_duopoly.py` module.

**Outline:**
1. [Setup and Notation](#1-setup-and-notation)
2. [Characteristic Equations](#2-characteristic-equations)
3. [H-Regime Option Value (Absorbing)](#3-h-regime-option-value)
4. [L-Regime Coupled ODE and Two-Term Solution](#4-l-regime-coupled-ode)
5. [Effective Revenue Coefficient $A_{\text{eff}}$](#5-effective-revenue-coefficient)
6. [Optimal Training Fraction (Proposition 1)](#6-optimal-training-fraction)
7. [Default Boundary and Faith-Based Survival (Proposition 2)](#7-default-boundary)
8. [Numerical Verification](#8-numerical-verification)

**References:**
- Guo, Miao & Morellec (2005), "Irreversible Investment with Regime Shifts"
- Dixit & Pindyck (1994), Chapter 6
- Leland (1994), "Corporate Debt Value, Bond Covenants, and Optimal Capital Structure"

In [ ]:
import sympy as sp
from IPython.display import Math
from IPython.display import display as ipy_display

sp.init_printing(use_latex="mathjax")


def show(label: str, expr):
    """Display a labeled equation."""
    ipy_display(Math(rf"\text{{{label}:}} \quad {sp.latex(expr)}"))

## 1. Setup and Notation

The demand shifter $X_t$ follows a regime-switching GBM:

$$
\frac{dX_t}{X_t} = \mu_s \, dt + \sigma_s \, dW_t, \quad s \in \{L, H\}
$$

Regime $L$ (pre-adoption) switches to regime $H$ (post-adoption, absorbing) at Poisson rate $\tilde{\lambda}$.

The firm invests irreversibly in capacity $K$ at cost $I(K) = c K^\gamma$ and allocates fraction $\phi$ to training, $(1-\phi)$ to inference.

In [ ]:
# Demand process
X = sp.Symbol("X", positive=True)
mu_L = sp.Symbol("mu_L", real=True)
mu_H = sp.Symbol("mu_H", real=True)
sigma_L = sp.Symbol("sigma_L", positive=True)
sigma_H = sp.Symbol("sigma_H", positive=True)
r = sp.Symbol("r", positive=True)
lam = sp.Symbol("lambda", positive=True)  # tilde{lambda}

# Technology
alpha = sp.Symbol("alpha", positive=True)
gamma = sp.Symbol("gamma", positive=True)
c = sp.Symbol("c", positive=True)
delta = sp.Symbol("delta", positive=True)
K = sp.Symbol("K", positive=True)
phi = sp.Symbol("phi", positive=True)

# Generic characteristic exponent
beta = sp.Symbol("beta")

# Named exponents
beta_H = sp.Symbol("beta_H", positive=True)
beta_L_plus = sp.Symbol("beta_L^+", positive=True)

# Option value coefficients
B_H = sp.Symbol("B_H", positive=True)
A1 = sp.Symbol("A_1")
C_coeff = sp.Symbol("C")

# Trigger
X_star = sp.Symbol("X^*", positive=True)

print("Symbols defined.")

## 2. Characteristic Equations

The option value in each regime satisfies an ODE whose homogeneous solutions are power functions $X^\beta$. The exponent $\beta$ solves a quadratic characteristic equation.

### H-regime (absorbing)

$$
\frac{\sigma_H^2}{2} \beta(\beta - 1) + \mu_H \beta - r = 0
$$

In [ ]:
# H-regime characteristic equation
char_H = sp.Rational(1, 2) * sigma_H**2 * beta * (beta - 1) + mu_H * beta - r
show("Q_H(\\beta)", sp.Eq(char_H, 0))

roots_H = sp.solve(char_H, beta)
print(f"\nRoots: beta_H^- = {roots_H[0]},  beta_H^+ = {roots_H[1]}")
show("\\beta_H^+", roots_H[1])

### L-regime (with regime switching)

The regime-switching term $\tilde{\lambda}[F_H(X) - F_L(X)]$ acts as an additional discount on $F_L$. The homogeneous ODE uses effective discount $r + \tilde{\lambda}$:

$$
\frac{\sigma_L^2}{2} \beta(\beta - 1) + \mu_L \beta - (r + \tilde{\lambda}) = 0
$$

In [ ]:
# L-regime characteristic equation
char_L = sp.Rational(1, 2) * sigma_L**2 * beta * (beta - 1) + mu_L * beta - (r + lam)
show("Q_L(\\beta)", sp.Eq(char_L, 0))

roots_L = sp.solve(char_L, beta)
print(f"\nRoots: beta_L^- = {roots_L[0]},  beta_L^+ = {roots_L[1]}")
show("\\beta_L^+", roots_L[1])

## 3. H-Regime Option Value

In the absorbing H-regime, the option value takes the standard real-options form:

$$
F_H(X) = B_H \cdot X^{\beta_H}, \quad X < X_H^*
$$

The installed value (post-investment) is:

$$
V_H(X, K) = A_H \cdot X \cdot (\phi K)^\alpha - \frac{\delta K}{r}, \quad A_H = \frac{1}{r - \mu_H}
$$

Value-matching and smooth-pasting at $X_H^*$ pin down $B_H$ and $X_H^*$.

In [ ]:
# Present-value multiplier
A_H = 1 / (r - mu_H)
show("A_H", A_H)

# Revenue coefficient and total cost
revenue_coeff = A_H * (phi * K) ** alpha
total_cost = c * K**gamma + delta * K / r

print("\nInstalled value at X*:")
show("V_H(X^*, K)", revenue_coeff * X_star - delta * K / r)
show("I(K) + \\delta K/r", total_cost)

In [ ]:
# VALUE-MATCHING: B_H * (X*)^{beta_H} = V_H(X*) - I(K)
# SMOOTH-PASTING: beta_H * B_H * (X*)^{beta_H - 1} = revenue_coeff

# From smooth-pasting:
B_H_expr = revenue_coeff * X_star ** (1 - beta_H) / beta_H
show("B_H", B_H_expr)

# Substituting into value-matching gives the trigger:
# revenue_coeff * X* / beta_H = revenue_coeff * X* - total_cost
# => revenue_coeff * X* * (1 - 1/beta_H) = total_cost
trigger_H = (beta_H / (beta_H - 1)) * total_cost / revenue_coeff
trigger_H_simplified = sp.simplify(trigger_H)

print("\nOptimal H-regime trigger:")
show("X_H^*", trigger_H_simplified)

print("\nThis has the standard real-options markup beta/(beta-1) > 1 over the")
print("Marshallian NPV=0 trigger, reflecting the option value of waiting.")

## 4. L-Regime Coupled ODE and Two-Term Solution

This is the critical derivation. The L-regime option value satisfies the HJB equation:

$$
\frac{1}{2} \sigma_L^2 X^2 F_L'' + \mu_L X F_L' + \tilde{\lambda}[F_H(X) - F_L(X)] - r F_L = 0
$$

Rearranging:

$$
\underbrace{\frac{1}{2} \sigma_L^2 X^2 F_L'' + \mu_L X F_L' - (r + \tilde{\lambda}) F_L}_{\text{Homogeneous operator}} + \underbrace{\tilde{\lambda} B_H X^{\beta_H}}_{\text{Non-homogeneous term}} = 0
$$

Since $F_H(X) = B_H X^{\beta_H}$, this is a second-order Euler ODE with a power-function forcing term.

In [ ]:
# The general solution has three parts:
# F_L(X) = A1 * X^{beta_L^+} + A2 * X^{beta_L^-} + C * X^{beta_H}
#
# A2 = 0 (boundary condition F_L(0) = 0 since beta_L^- < 0)
# C is the particular solution coefficient
# A1 is determined by boundary conditions at the trigger

# PARTICULAR SOLUTION: Try F_p = C * X^{beta_H}
# Substituting into the homogeneous operator:
# C * Q_L(beta_H) * X^{beta_H} + lambda * B_H * X^{beta_H} = 0

Q_L_at_beta_H = (
    sp.Rational(1, 2) * sigma_L**2 * beta_H * (beta_H - 1) + mu_L * beta_H - (r + lam)
)
show("Q_L(\\beta_H)", Q_L_at_beta_H)

C_expr = -lam * B_H / Q_L_at_beta_H
print("\nParticular solution coefficient:")
show("C", C_expr)

print(
    "\nNote: Q_L(beta_H) != 0 because beta_H solves Q_H(beta) = 0, not Q_L(beta) = 0."
)
print("The additional lambda in the discount ensures they differ.")

In [ ]:
# General solution (dropping A2 = 0 term):
F_L = A1 * X**beta_L_plus + C_coeff * X**beta_H

print("General solution for F_L(X):")
show("F_L(X)", F_L)

print("\nTwo terms:")
print("  1. A_1 * X^{beta_L^+} : homogeneous (L-regime dynamics, discount r+lambda)")
print("  2. C * X^{beta_H}     : particular (regime-switching possibility)")
print("\nThe paper's simplified form F_L = C * X^{beta_H} is valid only when A_1 = 0.")

### Value-matching and smooth-pasting in L

For a firm with installed value $V_L(X) = A_{\text{eff}} X - \delta K / r$, the boundary conditions at trigger $X^*$ are:

$$
\text{VM:} \quad A_1 (X^*)^{\beta_L^+} + C (X^*)^{\beta_H} = A_{\text{eff}} X^* - \frac{\delta K}{r} - c K^\gamma
$$

$$
\text{SP:} \quad A_1 \beta_L^+ (X^*)^{\beta_L^+ - 1} + C \beta_H (X^*)^{\beta_H - 1} = A_{\text{eff}}
$$

In [ ]:
A_eff_sym = sp.Symbol("A_{eff}", positive=True)

npv_at_trigger = A_eff_sym * X_star - delta * K / r - c * K**gamma

# Value-matching
vm = sp.Eq(
    A1 * X_star**beta_L_plus + C_coeff * X_star**beta_H,
    npv_at_trigger,
)
print("Value-matching:")
ipy_display(vm)

# Smooth-pasting
sp_eq = sp.Eq(
    A1 * beta_L_plus * X_star ** (beta_L_plus - 1)
    + C_coeff * beta_H * X_star ** (beta_H - 1),
    A_eff_sym,
)
print("\nSmooth-pasting:")
ipy_display(sp_eq)

In [ ]:
# Solve for A1 from smooth-pasting
A1_from_sp = sp.solve(sp_eq, A1)[0]
print("A_1 from smooth-pasting:")
show("A_1", A1_from_sp)

# Substitute into value-matching for the implicit trigger equation
trigger_eq = sp.simplify(vm.subs(A1, A1_from_sp))
print("\nImplicit trigger equation (substituting A_1 into VM):")
ipy_display(trigger_eq)

print("\nThis implicitly determines X* given (K, phi, hence A_eff).")
print("In general, it does NOT simplify to beta_H/(beta_H-1)")
print("* cost/A_eff. That formula is valid only when A_1 = 0.")

### When is $A_1 = 0$?

The homogeneous term $A_1 = 0$ when no interior trigger exists in regime $L$. This happens when the option premium ratio $\Phi_L = (1 - 1/\beta_L^+)/\alpha \geq 1$. Economically: the option to wait is so valuable that the firm never exercises in $L$ --- it waits for the regime switch.

In [ ]:
# Option premium ratio
Phi_L = (1 - 1 / beta_L_plus) / alpha
show("\\Phi_L", Phi_L)

print("\nInterior L-trigger exists when: 1/gamma < Phi_L < 1")
print("A_1 = 0 (simplified form valid) when: Phi_L >= 1")
print("\nFor baseline parameters (sigma_L=0.25, mu_L=0.01, r=0.12, lambda=0.10):")
print("  Effective discount = r + lambda = 0.22")
print("  beta_L^+ is large => Phi_L > 1 => A_1 = 0")
print("  The simplified F_L = C * X^{beta_H} IS valid at baseline.")

## 5. Effective Revenue Coefficient $A_{\text{eff}}$

The training-inference allocation creates a combined effective revenue coefficient that captures both $L$-regime inference revenue and $H$-regime continuation value:

$$
A_{\text{eff}}(\phi, K) = \frac{[(1-\phi)K]^\alpha}{r - \mu_L + \tilde{\lambda}} + \frac{\tilde{\lambda}}{r - \mu_L + \tilde{\lambda}} \cdot \frac{[\phi K]^\alpha}{r - \mu_H}
$$

The first term is discounted inference revenue; the second is the expected present value of $H$-regime training quality, weighted by the arrival rate.

In [ ]:
denom_L = r - mu_L + lam
A_H_pv = 1 / (r - mu_H)

inf_cap = (1 - phi) * K
tr_cap = phi * K

A_eff = inf_cap**alpha / denom_L + lam / denom_L * tr_cap**alpha * A_H_pv

show("A_{\\text{eff}}(\\phi, K)", A_eff)

In [ ]:
# Partial derivative w.r.t. phi (for optimal training fraction)
dA_dphi = sp.diff(A_eff, phi)
dA_dphi_simplified = sp.simplify(dA_dphi)
print("dA_eff / dphi:")
show("\\partial A_{\\text{eff}} / \\partial \\phi", dA_dphi_simplified)

# Partial derivative w.r.t. lambda (for faith-based survival)
dA_dlam = sp.diff(A_eff, lam)
dA_dlam_simplified = sp.simplify(dA_dlam)
print("\ndA_eff / dlambda:")
show("\\partial A_{\\text{eff}} / \\partial \\tilde{\\lambda}", dA_dlam_simplified)

### Sign of $\partial A_{\text{eff}} / \partial \tilde{\lambda}$

For the faith-based survival mechanism (Proposition 2), we need $\partial A_{\text{eff}} / \partial \tilde{\lambda} > 0$. This holds when the $H$-regime training value exceeds the $L$-regime inference value that is "discounted away" by the higher effective discount rate.

In [ ]:
# To check the sign, evaluate symbolically:
# dA_eff/dlam = d/dlam [ inf^alpha/(r-mu_L+lam) + lam/(r-mu_L+lam) * tr^alpha * A_H ]
#
# = -inf^alpha/(r-mu_L+lam)^2 + (r-mu_L)/(r-mu_L+lam)^2 * tr^alpha * A_H
#
# = 1/(r-mu_L+lam)^2 * [tr^alpha * A_H * (r - mu_L) - inf^alpha]
#
# This is positive when:
# tr^alpha * A_H * (r - mu_L) > inf^alpha
# i.e., when H-regime continuation value exceeds L-regime inference value

condition_positive = tr_cap**alpha * A_H_pv * (r - mu_L) - inf_cap**alpha
print("dA_eff/dlambda > 0  when the following expression is positive:")
show("\\text{Condition}", sp.simplify(condition_positive))

print("\nEconomically: the H-regime continuation value (weighted by r - mu_L)")
print("must exceed the L-regime inference value that is 'lost' to the")
print("higher effective discount.")

## 6. Optimal Training Fraction (Proposition 1)

The firm maximizes the option value factor $h(K, \phi) = A_{\text{eff}}^{\beta_H} / \text{cost}^{\beta_H - 1}$.

Since cost does not depend on $\phi$, the first-order condition reduces to:

$$
\frac{\partial A_{\text{eff}}}{\partial \phi} = 0
$$

The key insight is that $\alpha < 1$ generates Inada conditions at both boundaries, guaranteeing an interior solution.

In [ ]:
# FOC: dA_eff / dphi = 0
foc = sp.diff(A_eff, phi)
foc_simplified = sp.simplify(foc)

print("First-order condition for optimal phi:")
show("\\frac{\\partial A_{\\text{eff}}}{\\partial \\phi} = 0", sp.Eq(foc_simplified, 0))

In [ ]:
# Separate the two terms of the FOC to show the trade-off
# d/dphi [(1-phi)K]^alpha = -alpha * K * [(1-phi)K]^{alpha-1}
# d/dphi [phi*K]^alpha = alpha * K * [phi*K]^{alpha-1}

marginal_inference = -alpha * K * ((1 - phi) * K) ** (alpha - 1) / denom_L
marginal_training = lam / denom_L * alpha * K * (phi * K) ** (alpha - 1) * A_H_pv

print("Marginal cost of training (lost inference revenue):")
show("MC_{\\phi}", sp.simplify(marginal_inference))

print("\nMarginal benefit of training (H-regime value):")
show("MB_{\\phi}", sp.simplify(marginal_training))

print("\nAt the optimum, MC = MB (in absolute value).")

# Equating and solving for phi
foc_ratio = sp.Eq(marginal_training + marginal_inference, 0)
phi_solutions = sp.solve(foc_ratio, phi)
if phi_solutions:
    print("\nSolution for phi*:")
    for sol in phi_solutions:
        show("\\phi^*", sp.simplify(sol))

In [ ]:
# Verify Inada conditions
print("Boundary behavior (alpha < 1):")
print()
print("As phi -> 0+:")
print("  d/dphi [phi*K]^alpha = alpha*K^alpha * phi^{alpha-1} -> +inf")
print("  => dA_eff/dphi -> +inf (training marginal value infinite)")
print()
print("As phi -> 1-:")
print("  d/dphi [(1-phi)*K]^alpha = -alpha*K^alpha * (1-phi)^{alpha-1} -> -inf")
print("  => dA_eff/dphi -> -inf (inference marginal value infinite)")
print()
print("By the intermediate value theorem, an interior phi* in (0,1) exists.")
print("Concavity of A_eff in phi (from alpha < 1) ensures uniqueness.")

### Comparative statics of $\phi^*$

The optimal training fraction is increasing in $\tilde{\lambda}$: more optimistic firms allocate more to training.

In [ ]:
# Implicit function theorem:
# d phi*/d lam = -(d^2 A/dphi dlam) / (d^2 A/dphi^2)
# Since d^2 A/dphi^2 < 0 (concavity),
# sign of dphi*/dlam = sign of d^2A/dphi dlam

cross_deriv = sp.diff(A_eff, phi, lam)
cross_simplified = sp.simplify(cross_deriv)

print("Cross-partial d^2 A_eff / (dphi dlambda):")
latex_cross = (
    "\\frac{\\partial^2 A_{\\text{eff}}}"
    "{\\partial \\phi \\, \\partial \\tilde{\\lambda}}"
)
show(latex_cross, cross_simplified)

# Second derivative w.r.t. phi (concavity)
second_deriv = sp.diff(A_eff, phi, phi)
second_simplified = sp.simplify(second_deriv)

print("\nSecond derivative d^2 A_eff / dphi^2:")
show(
    "\\frac{\\partial^2 A_{\\text{eff}}}{\\partial \\phi^2}",
    second_simplified,
)

print("\nSince alpha < 1: d^2A/dphi^2 < 0 (concave).")
print("The cross-partial is positive when the H-regime training")
print("value exceeds L-regime inference, so dphi*/dlambda > 0.")

## 7. Default Boundary and Faith-Based Survival (Proposition 2)

Following Leland (1994), equity holders optimally default when $X$ falls to the endogenous boundary:

$$
X_D = \frac{\beta^-}{\beta^- - 1} \cdot \frac{c_D/r + \delta K / r}{A_{\text{eff}}}
$$

where $\beta^- < 0$ is the negative characteristic root and $c_D$ is the coupon payment.

Since $\beta^- < 0$, we have $\beta^-/(\beta^- - 1) \in (0, 1)$, so $X_D$ is below the naive break-even level.

In [ ]:
beta_neg = sp.Symbol("beta^-", negative=True)
c_D = sp.Symbol("c_D", positive=True)

X_D = (beta_neg / (beta_neg - 1)) * (c_D / r + delta * K / r) / A_eff_sym

print("Endogenous default boundary:")
show("X_D", X_D)

In [ ]:
# Comparative statics
dXD_dA = sp.diff(X_D, A_eff_sym)
dXD_dcD = sp.diff(X_D, c_D)

print("Comparative statics of X_D:")
print()
show("\\partial X_D / \\partial A_{\\text{eff}}", sp.simplify(dXD_dA))
print("  Sign: < 0 (higher effective revenue LOWERS default boundary)")
print()
show("\\partial X_D / \\partial c_D", sp.simplify(dXD_dcD))
print("  Sign: > 0 (higher coupon RAISES default boundary)")

In [ ]:
print("=" * 70)
print("FAITH-BASED SURVIVAL MECHANISM (Proposition 2)")
print("=" * 70)
print()
print("Chain rule: dX_D/dlambda = (dX_D/dA_eff) * (dA_eff/dlambda)")
print()
print("Since:")
print("  (1) dX_D/dA_eff < 0  (higher revenue lowers default boundary)")
print("  (2) dA_eff/dlambda > 0  (when H-regime value > L-regime value)")
print()
print("=> dX_D/dlambda < 0")
print()
print("Higher arrival rate LOWERS the default boundary.")
print("The hope of transformative AI literally keeps the firm alive.")
print()
print("Training investment raises A_eff through the second term")
print("(H-regime continuation value), which lowers X_D.")
print("But training also reduces the first term (L-regime inference),")
print("weakening current cash flow.")
print()
print("The mechanism works only when the continuation-value channel")
print("dominates the current-cash-flow channel.")

## 8. Numerical Verification

We verify the symbolic derivations against the numerical implementation in `base_model.py`.

In [ ]:
from ai_lab_investment.models.base_model import SingleFirmModel
from ai_lab_investment.models.parameters import ModelParameters

params = ModelParameters()
model = SingleFirmModel(params)

print("Baseline parameters:")
print(f"  r={params.r}, mu_L={params.mu_L}, mu_H={params.mu_H}")
print(f"  sigma_L={params.sigma_L}, sigma_H={params.sigma_H}")
print(f"  lam={params.lam}, alpha={params.alpha}, gamma={params.gamma}")
print(f"  beta_H={params.beta_H:.6f}, beta_L={params.beta_L:.6f}")

In [ ]:
# Verify characteristic roots
p = params

# H-regime: (sigma_H^2/2)*b*(b-1) + mu_H*b - r = 0
Q_H_val = 0.5 * p.sigma_H**2 * p.beta_H * (p.beta_H - 1) + p.mu_H * p.beta_H - p.r
print(f"Q_H(beta_H) = {Q_H_val:.2e}  (should be ~0)")

# L-regime: (sigma_L^2/2)*b*(b-1) + mu_L*b - (r+lam) = 0
Q_L_val = (
    0.5 * p.sigma_L**2 * p.beta_L * (p.beta_L - 1) + p.mu_L * p.beta_L - (p.r + p.lam)
)
print(f"Q_L(beta_L) = {Q_L_val:.2e}  (should be ~0)")

In [ ]:
# Verify particular solution coefficient C
C_code = model._particular_solution_coeff()
_, _, B_H_code = model._solve_regime_H()

Q_L_at_bH = (
    0.5 * p.sigma_L**2 * p.beta_H * (p.beta_H - 1) + p.mu_L * p.beta_H - (p.r + p.lam)
)
C_formula = -p.lam * B_H_code / Q_L_at_bH

print(f"C from code:    {C_code:.10f}")
print(f"C from formula: {C_formula:.10f}")
print(f"Match: {abs(C_code - C_formula) < 1e-12}")

In [ ]:
# Verify that A1 = 0 at baseline (no interior L-trigger)
phi_L = (1.0 - 1.0 / p.beta_L) / p.alpha
has_L_trigger = model.has_interior_trigger("L")

print(f"Option premium ratio Phi_L = {phi_L:.4f}")
print(f"Interior L-trigger exists: {has_L_trigger}")
print(f"Phi_L >= 1: {phi_L >= 1.0}")
valid = "VALID" if phi_L >= 1.0 else "INVALID"
print(f"=> Simplified F_L = C * X^{{beta_H}} is {valid}")

In [ ]:
# Verify option value structure
test_X_vals = [0.01, 0.05, 0.10, 0.20]

print("Option value verification (F_L = C * X^beta_H):")
print(f"{'X':>8s}  {'F_L(code)':>14s}  {'C*X^bH':>14s}  {'Match':>8s}")
print("-" * 50)

for test_X in test_X_vals:
    F_code = model.option_value_L(test_X)
    F_formula = C_code * test_X**p.beta_H
    match = abs(F_code - F_formula) < 1e-10
    print(f"{test_X:8.3f}  {F_code:14.8f}  {F_formula:14.8f}  {match!s:>8s}")

In [ ]:
# Verify H-regime trigger and capacity
X_H, K_H, _ = model._solve_regime_H()

# Verify trigger formula: X* = beta_H/(beta_H-1) * cost / (A_H * K^alpha)
markup = p.beta_H / (p.beta_H - 1)
cost = p.delta * K_H / p.r + p.c * K_H**p.gamma
rev_coeff = p.A_H * K_H**p.alpha
X_H_formula = markup * cost / rev_coeff

print(f"H-regime trigger from code:    X_H* = {X_H:.8f}")
print(f"H-regime trigger from formula: X_H* = {X_H_formula:.8f}")
print(f"Match: {abs(X_H - X_H_formula) < 1e-6}")
print(f"\nOptimal capacity: K_H* = {K_H:.6f}")
print(f"Option premium markup: beta_H/(beta_H-1) = {markup:.4f}")

In [ ]:
# Verify optimal training fraction
X_opt, K_opt, phi_opt = model.optimal_trigger_capacity_phi()

print("Optimal joint solution:")
print(f"  X*   = {X_opt:.6f}")
print(f"  K*   = {K_opt:.6f}")
print(f"  phi* = {phi_opt:.6f}")

# Verify FOC: dA_eff/dphi = 0 at phi*
eps = 1e-6
A_eff_plus = model._effective_revenue_coeff_single(phi_opt + eps, K_opt)
A_eff_minus = model._effective_revenue_coeff_single(phi_opt - eps, K_opt)
numerical_deriv = (A_eff_plus - A_eff_minus) / (2 * eps)

print(f"\nFOC check: dA_eff/dphi at phi* = {numerical_deriv:.2e} (should be ~0)")

In [ ]:
# Verify comparative static: phi* increasing in lambda
lambdas = [0.05, 0.10, 0.15, 0.20, 0.30]
phi_stars = []

print("phi* as a function of lambda (should be increasing):")
print(f"{'lambda':>8s}  {'phi*':>8s}  {'X*':>10s}  {'K*':>10s}")
print("-" * 40)

for lam_val in lambdas:
    p_i = params.with_param(lam=lam_val)
    m_i = SingleFirmModel(p_i)
    try:
        X_i, K_i, phi_i = m_i.optimal_trigger_capacity_phi()
        phi_stars.append(phi_i)
        print(f"{lam_val:8.2f}  {phi_i:8.4f}  {X_i:10.4f}  {K_i:10.4f}")
    except RuntimeError:
        print(f"{lam_val:8.2f}  (no solution)")

if len(phi_stars) > 1:
    is_increasing = all(
        phi_stars[i] <= phi_stars[i + 1] for i in range(len(phi_stars) - 1)
    )
    print(f"\nphi* monotonically increasing in lambda: {is_increasing}")

## Summary

This notebook verified:

1. **Characteristic equations**: Both H- and L-regime roots satisfy their respective quadratic equations.
2. **H-regime option value**: The trigger formula $X_H^* = \frac{\beta_H}{\beta_H - 1} \cdot \frac{\text{cost}}{A_H K^\alpha}$ matches the numerical solver.
3. **L-regime two-term solution**: The particular solution coefficient $C = -\tilde{\lambda} B_H / Q_L(\beta_H)$ matches the code. At baseline parameters, $A_1 = 0$ (no interior L-trigger), validating the simplified form $F_L = C X^{\beta_H}$.
4. **Effective revenue coefficient**: $A_{\text{eff}}$ correctly combines inference and training values.
5. **Optimal training fraction**: The FOC $\partial A_{\text{eff}} / \partial \phi = 0$ holds at the numerically computed $\phi^*$, and $\phi^*$ is increasing in $\tilde{\lambda}$.
6. **Default boundary**: The faith-based survival mechanism ($dX_D/d\tilde{\lambda} < 0$) follows from the chain rule on $A_{\text{eff}}$.